# 03_dice_guardrails

Fit the DiCE-compatible risk model, generate counterfactuals, and apply guardrails (direction, realism, binary snapping, model re-check). Demonstrates that raw DiCE produces infeasible/direction-reversed recourse and that the highest-risk group has no feasible recourse, so counterfactuals are meaningful only for the borderline group.

In [1]:
# 03_dice_guardrails.ipynb
# Generate counterfactual recourse with DiCE on the prospective risk model,
# then enforce guardrails to keep only clinically feasible, direction-consistent
# recommendations.

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import dice_ml
from dice_ml import Dice

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")

htn = pd.read_parquet(os.path.join(DATA, "htn_analysis.parquet"))
htn["female"] = (htn["SEX"] == 2).astype(float)
FEATS = ["BMI", "age", "female", "smoke_cur", "exer_reg", "walk_days"]

# All-float dtype avoids a known DiCE 0.12 / pandas casting conflict
X = htn[FEATS].astype(float)
y = htn["incident"].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25,
                                      random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=30,
                             class_weight="balanced", random_state=42).fit(Xtr, ytr)

train_df = Xtr.copy(); train_df["incident"] = ytr.values
d   = dice_ml.Data(dataframe=train_df, continuous_features=FEATS, outcome_name="incident")
mdl = dice_ml.Model(model=clf, backend="sklearn")
exp = Dice(d, mdl, method="random")
print("DiCE explainer ready.")

DiCE explainer ready.


In [2]:
# Guardrail function: enforce direction/realism, snap binaries, re-check model.
def guardrail(orig, cf_row):
    o = orig; c = cf_row.copy()
    # snap binary/count features to valid values
    for b in ["smoke_cur", "exer_reg"]:
        c[b] = int(round(np.clip(c[b], 0, 1)))
    c["walk_days"] = int(round(np.clip(c["walk_days"], 0, 7)))
    # BMI: reduction only, floor at 18.5 and at most 15% loss
    bmi_floor = max(18.5, o["BMI"] * 0.85)
    if c["BMI"] > o["BMI"]:
        c["BMI"] = o["BMI"]
    c["BMI"] = max(c["BMI"], bmi_floor)
    # smoking: never suggest starting (only cessation direction)
    if o["smoke_cur"] == 0:
        c["smoke_cur"] = 0
    # exercise: never suggest stopping (only increase direction)
    if o["exer_reg"] == 1:
        c["exer_reg"] = 1
    # walking: no reduction
    if c["walk_days"] < o["walk_days"]:
        c["walk_days"] = int(o["walk_days"])
    changed = (abs(c["BMI"] - o["BMI"]) > 0.1) or (c["smoke_cur"] != o["smoke_cur"]) \
              or (c["exer_reg"] != o["exer_reg"]) or (c["walk_days"] != o["walk_days"])
    return c, changed

In [3]:
# Counterfactuals are meaningful for the BORDERLINE risk group.
# The highest-risk group (elderly + obese) is dominated by age (immutable),
# so no feasible recourse exists there.
Xte2 = Xte.copy()
Xte2["risk"] = clf.predict_proba(Xte)[:, 1]
targets = Xte2[(Xte2["risk"] >= 0.35) & (Xte2["risk"] <= 0.55)] \
              .sort_values("risk", ascending=False).head(6)

cf = exp.generate_counterfactuals(
    targets[FEATS], total_CFs=6, desired_class=0,
    features_to_vary=["BMI", "smoke_cur", "exer_reg", "walk_days"],
    permitted_range={"BMI": [18.5, 45.0], "smoke_cur": [0, 1],
                     "exer_reg": [0, 1], "walk_days": [0, 7]})

n_valid = 0
for i in range(len(targets)):
    o = targets[FEATS].iloc[i]
    print(f"[case {i+1}] BMI={o['BMI']:.1f} age={o['age']:.0f} "
          f"smoke={int(o['smoke_cur'])} exer={int(o['exer_reg'])} "
          f"walk={int(o['walk_days'])} | risk={targets['risk'].iloc[i]:.2f}")
    cfdf = cf.cf_examples_list[i].final_cfs_df
    if cfdf is None:
        print("   no counterfactual found"); continue
    seen = set()
    for _, row in cfdf.iterrows():
        c, changed = guardrail(o, row)
        if not changed:
            continue
        xc = np.array([[c["BMI"], o["age"], o["female"],
                        c["smoke_cur"], c["exer_reg"], c["walk_days"]]])
        if clf.predict(xc)[0] != 0:      # must still cross to non-incident
            continue
        key = (round(c["BMI"], 1), c["smoke_cur"], c["exer_reg"], c["walk_days"])
        if key in seen:
            continue
        seen.add(key)
        parts = []
        if abs(c["BMI"] - o["BMI"]) > 0.1:
            parts.append(f"BMI {o['BMI']:.1f}->{c['BMI']:.1f}")
        if c["exer_reg"] > o["exer_reg"]:
            parts.append("start regular exercise")
        if c["walk_days"] > o["walk_days"]:
            parts.append(f"walk {int(o['walk_days'])}->{c['walk_days']} d/wk")
        print("   option:", ", ".join(parts)); n_valid += 1
print("total valid recourse:", n_valid)

  0%|          | 0/6 [00:00<?, ?it/s]

 17%|█▋        | 1/6 [00:00<00:01,  2.56it/s]

 33%|███▎      | 2/6 [00:00<00:01,  2.07it/s]

 50%|█████     | 3/6 [01:03<01:26, 28.75s/it]

 67%|██████▋   | 4/6 [01:21<00:49, 24.52s/it]

...

100%|██████████| 6/6 [01:22<00:00, 10.66s/it]

100%|██████████| 6/6 [01:22<00:00, 13.74s/it]

[case 1] BMI=25.8 age=54 smoke=0 exer=1 walk=7 | risk=0.55
   option: BMI 25.8->21.9
   option: BMI 25.8->22.3
   option: BMI 25.8->24.1
[case 2] BMI=24.8 age=56 smoke=0 exer=1 walk=6 | risk=0.55
   option: BMI 24.8->21.1
[case 3] BMI=20.1 age=85 smoke=0 exer=0 walk=2 | risk=0.55
   option: BMI 20.1->19.8
   option: BMI 20.1->19.7


[case 4] BMI=21.4 age=67 smoke=0 exer=1 walk=3 | risk=0.55
   option: BMI 21.4->21.2
   option: BMI 21.4->18.9, walk 3->4.0 d/wk
[case 5] BMI=22.0 age=66 smoke=0 exer=1 walk=4 | risk=0.55
   option: BMI 22.0->21.0
   option: BMI 22.0->18.9
   option: BMI 22.0->19.6
   option: BMI 22.0->19.2
[case 6] BMI=22.2 age=67 smoke=0 exer=1 walk=3 | risk=0.55
   option: BMI 22.2->19.0
   option: BMI 22.2->19.4


   option: BMI 22.2->19.5
   option: BMI 22.2->19.1
   option: BMI 22.2->21.2
   option: BMI 22.2->19.6
total valid recourse: 18
